In [1]:
# Cell 1 - Install required packages (run once)
!pip install -qU \
sentence-transformers \
faiss-cpu \
"transformers==4.44.2" \
"accelerate>=0.33.0" \
datasets \
sentencepiece


In [2]:
# Cell 2 - Imports + basic setup
import os
import json
from typing import List, Dict, Any
from pathlib import Path
from tqdm.auto import tqdm

import numpy as np

# sentence-transformers for embeddings
from sentence_transformers import SentenceTransformer

# FAISS for vector index
import faiss

# transformers for generation
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, pipeline

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [4]:
!ls /content/drive/MyDrive/


'Air force trip'    DatasetTrainingV2	       'Gaming night'
'Bodim athl'	   'D.M.T.M Dissanayaka..pdf'  'Rathmulu kanda 2024 01 01'
'Colab Notebooks'   Dolosbage		       'Tharindu Dissanayaka..pdf'
 datasets	    FDM_MiniProject_Dataset    'Tharindu Dissanayaka.pdf'


In [5]:
KB_FILE = "/content/drive/MyDrive/datasets/miriad_balanced_300.json"

KB_PATHS_TRY = ["/content/drive/MyDrive/datasets/miriad_balanced_300.json"]

EMBEDDING_MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"
GENERATOR_MODEL_NAME = "google/flan-t5-base"
INDEX_PATH = "kb_faiss.index"
META_PATH = "kb_metadata.json"
EMBEDDING_DIM = 384
TOP_K = 5
DEVICE = 0


In [6]:
# Cell 4 - Helper: load your knowledge base flexibly
def load_kb(paths_try: List[str]) -> List[Dict[str, Any]]:
    """
    Try to load a JSON/JSONL file. Returns list of dicts.
    Acceptable item keys (recommended): 'id', 'question', 'answer', 'specialty', 'source'
    If your items are plain Q/A pairs, this will still work.
    """
    path = None
    for p in paths_try:
        if os.path.exists(p):
            path = p
            break
    if path is None:
        raise FileNotFoundError(f"Could not find knowledge base. Tried: {paths_try}")

    items = []
    # try JSON lines
    with open(path, "r", encoding="utf-8") as f:
        text = f.read().strip()
        if text.startswith("["):
            # JSON array
            data = json.loads(text)
            if isinstance(data, list):
                items = data
            else:
                raise ValueError("JSON root is not a list.")
        else:
            # try parse as JSONL
            lines = text.splitlines()
            try:
                for ln in lines:
                    if ln.strip():
                        items.append(json.loads(ln))
            except Exception as e:
                # fallback: try to parse as a single JSON object with a key containing list
                data = json.loads(text)
                if isinstance(data, dict):
                    # find first list-like value
                    for v in data.values():
                        if isinstance(v, list):
                            items = v
                            break
                    if not items:
                        raise ValueError("Couldn't find a list-of-items structure in JSON.")
                else:
                    raise e
    # normalize: ensure each item is a dict with 'id', 'text' fields
    normalized = []
    for i, it in enumerate(items):
        if not isinstance(it, dict):
            continue
        nid = it.get("id", f"doc_{i}")
        # Build a text field combining question and answer and any metadata
        parts = []
        if it.get("question"): parts.append("Q: " + str(it.get("question")).strip())
        if it.get("answer"): parts.append("A: " + str(it.get("answer")).strip())
        # keep other fields as metadata
        meta = {k: v for k, v in it.items() if k not in ("question", "answer")}
        text = " ".join(parts).strip()
        if not text:
            # maybe the JSON uses 'text' or 'content'
            text = it.get("text") or it.get("content") or ""
        if not text:
            # if still empty, skip
            continue
        normalized.append({"id": nid, "text": text, "meta": meta})
    return normalized

# Try to load KB now
kb = load_kb(KB_PATHS_TRY)
print(f"Loaded {len(kb)} documents from knowledge base. Example doc:")
print(kb[0] if len(kb) else "KB is empty")

Loaded 2400 documents from knowledge base. Example doc:
{'id': 'doc_0', 'text': 'Q: How do patients with significant pulmonary manifestations of BHD seek appropriate medical care? A: Patients with significant pulmonary manifestations of BHD often seek a pulmonary team that is confident in managing BHD lungs. However, there are very few pulmonologists/thoracic surgeons in the United States with clinical experience in managing BHD lungs. The LAM Foundation supports clinics that may be a beneficial clinical resource for these patients.', 'meta': {'qa_id': '38_18980821_0_2', 'paper_id': '18980821', 'paper_url': 'https://api.semanticscholar.org/CorpusID:18980821', 'paper_title': 'Birt–Hogg–Dubé: beyond the clinical manifestations', 'passage_text': "Individuals and their families, who live with Birt-HoggDubé (BHD), experience a varied medical experience. For some patients, life may be relatively simple involving mild manifestations with periodic kidney imaging, while for others, life may be 

In [7]:
# Cell 5 - Build or load FAISS index
def build_faiss_index(docs: List[Dict[str, Any]],
                      embedding_model_name: str = EMBEDDING_MODEL_NAME,
                      index_path: str = INDEX_PATH,
                      meta_path: str = META_PATH,
                      embedding_dim: int = EMBEDDING_DIM) -> faiss.Index:
    # 1) load embedding model
    print("Loading embedding model:", embedding_model_name)
    embedder = SentenceTransformer(embedding_model_name)

    # 2) build embeddings in batches
    texts = [d["text"] for d in docs]
    print(f"Computing embeddings for {len(texts)} docs...")
    embeddings = embedder.encode(texts, show_progress_bar=True, convert_to_numpy=True, normalize_embeddings=True)

    # 3) create faiss index (IndexFlatIP since we normalized embeddings => inner product equals cosine)
    index = faiss.IndexFlatIP(embedding_dim)
    index.add(embeddings.astype("float32"))
    print("FAISS index size:", index.ntotal)

    # 4) save index and metadata mapping
    faiss.write_index(index, index_path)
    # save metadata: ids and original texts + meta
    metadata = [{"id": d["id"], "text": d["text"], "meta": d["meta"]} for d in docs]
    with open(meta_path, "w", encoding="utf-8") as f:
        json.dump(metadata, f, ensure_ascii=False, indent=2)
    print(f"Saved index to {index_path} and metadata to {meta_path}")
    return index

# Only build if index doesn't exist; otherwise load
if not os.path.exists(INDEX_PATH) or not os.path.exists(META_PATH):
    index = build_faiss_index(kb)
else:
    print("Loading existing FAISS index and metadata...")
    index = faiss.read_index(INDEX_PATH)
    with open(META_PATH, "r", encoding="utf-8") as f:
        metadata = json.load(f)
    print("Index loaded. Total docs in index:", index.ntotal)

Loading existing FAISS index and metadata...
Index loaded. Total docs in index: 2400


In [8]:
# Cell 6 - Prepare generator pipeline
print("Loading generator model:", GENERATOR_MODEL_NAME)
tokenizer = AutoTokenizer.from_pretrained(GENERATOR_MODEL_NAME, use_fast=True)
model = AutoModelForSeq2SeqLM.from_pretrained(GENERATOR_MODEL_NAME)


Loading generator model: google/flan-t5-base


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


In [9]:
import transformers, torch
print("transformers:", transformers.__version__)
print("torch:", torch.__version__)


transformers: 4.44.2
torch: 2.10.0+cu128


In [10]:
rag_generator = pipeline(
    "text2text-generation",
    model=model,
    tokenizer=tokenizer,
    device=DEVICE
)
print("Generator ready.")


Generator ready.


In [11]:
# ✅ Separate LLM for Judge (different from generator)
JUDGE_MODEL_NAME = "google/flan-t5-large"   # best if you have GPU
# If you get OOM, fallback:
# JUDGE_MODEL_NAME = "google/flan-t5-small"

judge_tokenizer = AutoTokenizer.from_pretrained(JUDGE_MODEL_NAME, use_fast=True)
judge_model = AutoModelForSeq2SeqLM.from_pretrained(JUDGE_MODEL_NAME)

judge_llm = pipeline(
    "text2text-generation",
    model=judge_model,
    tokenizer=judge_tokenizer,
    device=DEVICE
)

print("✅ Judge model ready:", JUDGE_MODEL_NAME)

✅ Judge model ready: google/flan-t5-large


In [12]:
def llm_as_judge(claim: str, evidence: str, max_length: int = 32):
    """
    Independent LLM Judge (separate model).
    Outputs strictly: SUPPORTED or UNSUPPORTED
    """

    if evidence is None or str(evidence).strip() == "":
        return {
            "claim": claim,
            "evidence": evidence,
            "label": "UNSUPPORTED",
            "raw_output": "NO_EVIDENCE"
        }

    prompt = (
        "You are a strict medical fact-checking judge.\n"
        "Decide if the CLAIM is fully supported by the EVIDENCE.\n"
        "Rules:\n"
        "1) If the evidence does not explicitly support the claim, answer UNSUPPORTED.\n"
        "2) Do NOT use outside knowledge.\n"
        "3) Answer with exactly one word: SUPPORTED or UNSUPPORTED.\n\n"
        f"EVIDENCE:\n{evidence}\n\n"
        f"CLAIM:\n{claim}\n\n"
        "VERDICT:"
    )

    out = judge_llm(
        prompt,
        max_length=max_length,
        do_sample=False
    )[0]["generated_text"].strip().upper()

    # Safe label extraction
    token = out.split()[0] if out else "UNSUPPORTED"
    label = "SUPPORTED" if token == "SUPPORTED" else "UNSUPPORTED"

    return {
        "claim": claim,
        "evidence": evidence,
        "label": label,
        "raw_output": out
    }

In [13]:
# Cell 7 - Retrieval + Generation functions
# Load the embedding model again (for queries)
embedder = SentenceTransformer(EMBEDDING_MODEL_NAME)

# load metadata if not already present
if 'metadata' not in globals():
    with open(META_PATH, "r", encoding="utf-8") as f:
        metadata = json.load(f)

def retrieve(query: str, top_k: int = TOP_K):
    q_emb = embedder.encode([query], convert_to_numpy=True, normalize_embeddings=True)
    # search
    D, I = index.search(q_emb.astype("float32"), top_k)
    results = []
    for score, idx in zip(D[0], I[0]):
        if idx < 0 or idx >= len(metadata):
            continue
        entry = metadata[idx]
        results.append({"score": float(score), "id": entry["id"], "text": entry["text"], "meta": entry.get("meta", {})})
    return results

def construct_prompt(query: str, retrieved: List[Dict[str, Any]]):
    """
    Build a simple prompt for the generator: instructions + concatenated contexts.
    You can change format as you like (more system-like, or use templates).
    """
    context_parts = []
    for i, r in enumerate(retrieved, start=1):
        # include id and text; keep it short
        context_parts.append(f"Source {i} (id:{r['id']}): {r['text']}")
    context_block = "\n\n".join(context_parts)
    prompt = (
        "You are a helpful medical assistant. Answer the user's question using ONLY the information in the provided sources. "
        "If the answer is not contained in the sources, say you don't know and suggest possible next steps.\n\n"
        f"CONTEXT:\n{context_block}\n\n"
        f"QUESTION: {query}\n\n"
        "ANSWER:"
    )
    return prompt

def answer_question(query: str, top_k: int = TOP_K, gen_max_length: int = 256, temperature: float = 0.0):
    retrieved = retrieve(query, top_k=top_k)
    if not retrieved:
        return {"answer": "No relevant documents found in KB.", "retrieved": []}
    prompt = construct_prompt(query, retrieved)
    # generate
    outputs = rag_generator(prompt, max_length=gen_max_length, do_sample=(temperature>0.0), temperature=temperature)
    # pipeline returns list of dicts: [{'generated_text': '...'}]
    generated_text = outputs[0]["generated_text"].strip()
    return {"answer": generated_text, "retrieved": retrieved, "prompt": prompt, "raw_generation": outputs}

In [14]:
# =========================
# NEW CELL: Domain / Scope Check (Passage-level index)
# Keeps your current RAG + verification pipeline untouched
# SciSpacy optional: falls back to regex if nlp_sci is unavailable
# =========================

import re

SCOPE_INDEX_PATH = "scope_faiss.index"
SCOPE_META_PATH  = "scope_metadata.json"
SCOPE_TOP_K      = 10

# Thresholds (tune later using in-domain vs OOD questions)
SCOPE_MIN_TOP1     = 0.35
SCOPE_MIN_AVG_TOPK = 0.30
SCOPE_MIN_COHESION = 0.20

_SENT_SPLIT = re.compile(r'(?<=[.!?])\s+')

def _split_sentences(text: str) -> List[str]:
    """
    Sentence splitting with graceful fallback:
    - If nlp_sci exists and works -> SciSpacy sentence splitter
    - Else -> regex splitter
    """
    # Try SciSpacy if available
    try:
        if "nlp_sci" in globals() and nlp_sci is not None:
            doc = nlp_sci(text)
            sents = [s.text.strip() for s in doc.sents if s.text.strip()]
            if len(sents) > 0:
                return sents
    except Exception:
        pass

    # Fallback: regex splitter
    return [s.strip() for s in _SENT_SPLIT.split(text) if s.strip()]

def chunk_into_passages(text: str, max_sentences: int = 3) -> List[str]:
    """
    Split doc text into small passages (2–4 sentences each is ideal).
    Uses SciSpacy if available; otherwise regex.
    """
    sents = _split_sentences(text)
    chunks = []
    for i in range(0, len(sents), max_sentences):
        chunk = " ".join(sents[i:i+max_sentences]).strip()
        if chunk:
            chunks.append(chunk)
    return chunks

def build_scope_index_from_kb(kb_docs: List[Dict[str, Any]],
                              embedder,
                              index_path: str = SCOPE_INDEX_PATH,
                              meta_path: str = SCOPE_META_PATH,
                              max_sentences_per_passage: int = 3):
    """
    Build a separate FAISS index where each vector = one passage chunk.
    This is ONLY for domain checking.
    """
    passages = []
    scope_meta = []

    for d in kb_docs:
        doc_id = d.get("id")
        doc_text = d.get("text", "")
        doc_meta = d.get("meta", {})

        chunks = chunk_into_passages(doc_text, max_sentences=max_sentences_per_passage)

        for j, ch in enumerate(chunks):
            passages.append(ch)
            scope_meta.append({
                "doc_id": doc_id,
                "passage_id": f"{doc_id}_p{j}",
                "text": ch,
                "meta": doc_meta
            })

    if len(passages) == 0:
        raise ValueError("Scope index build failed: no passages were created. Check KB content.")

    print(f"🔧 Building scope index with {len(passages)} passages...")
    embs = embedder.encode(passages, convert_to_numpy=True, normalize_embeddings=True)

    scope_index = faiss.IndexFlatIP(EMBEDDING_DIM)
    scope_index.add(embs.astype("float32"))

    faiss.write_index(scope_index, index_path)
    with open(meta_path, "w", encoding="utf-8") as f:
        json.dump(scope_meta, f, ensure_ascii=False, indent=2)

    print(f"✅ Scope index saved: {index_path} | meta saved: {meta_path}")
    return scope_index, scope_meta

# Load or build scope index
if (not os.path.exists(SCOPE_INDEX_PATH)) or (not os.path.exists(SCOPE_META_PATH)):
    scope_index, scope_meta = build_scope_index_from_kb(kb, embedder)
else:
    scope_index = faiss.read_index(SCOPE_INDEX_PATH)
    with open(SCOPE_META_PATH, "r", encoding="utf-8") as f:
        scope_meta = json.load(f)
    print(f"✅ Loaded scope index. Passages in scope index: {scope_index.ntotal}")

def scope_check_similarity(query: str, top_k: int = SCOPE_TOP_K):
    q_emb = embedder.encode([query], convert_to_numpy=True, normalize_embeddings=True)
    D, I = scope_index.search(q_emb.astype("float32"), top_k)

    scores = [float(s) for s in D[0]] if len(D) else []
    idxs = [int(i) for i in I[0]] if len(I) else []

    top1 = scores[0] if scores else 0.0
    avgk = (sum(scores) / len(scores)) if scores else 0.0

    return top1, avgk, scores, idxs

def scope_check_cohesion(idxs: List[int], k: int = 5):
    """
    Measures how consistent the top retrieved passages are with each other.
    Low cohesion => scattered retrieval => likely OOD / unclear.
    """
    idxs = [i for i in idxs[:k] if (0 <= i < len(scope_meta))]
    if len(idxs) < 2:
        return 0.0

    texts = [scope_meta[i]["text"] for i in idxs]
    embs = embedder.encode(texts, convert_to_numpy=True, normalize_embeddings=True)

    sims = []
    for i in range(len(embs)):
        for j in range(i+1, len(embs)):
            sims.append(float(np.dot(embs[i], embs[j])))

    return float(sum(sims) / len(sims)) if sims else 0.0

def domain_check(query: str,
                 min_top1: float = SCOPE_MIN_TOP1,
                 min_avg_topk: float = SCOPE_MIN_AVG_TOPK,
                 min_cohesion: float = SCOPE_MIN_COHESION,
                 top_k: int = SCOPE_TOP_K):
    """
    Returns (in_domain: bool, info: dict)
    Uses passage-level similarity + cohesion as domain relativity gate.
    """
    top1, avgk, scores, idxs = scope_check_similarity(query, top_k=top_k)

    # Similarity gate
    if (top1 < min_top1) or (avgk < min_avg_topk):
        return False, {
            "reason": "low_similarity",
            "top1": top1,
            "avg_topk": avgk,
            "scores_top5": scores[:5]
        }

    # Cohesion gate (only if similarity looks okay)
    cohesion = scope_check_cohesion(idxs, k=min(5, top_k))
    if cohesion < min_cohesion:
        return False, {
            "reason": "low_cohesion",
            "top1": top1,
            "avg_topk": avgk,
            "cohesion": cohesion,
            "scores_top5": scores[:5]
        }

    return True, {
        "reason": "in_domain",
        "top1": top1,
        "avg_topk": avgk,
        "cohesion": cohesion,
        "scores_top5": scores[:5]
    }

def run_pipeline_with_domain_gate(query: str):
    """
    This wrapper ensures your existing pipeline runs ONLY if domain_check passes.
    Your current answer_question + verification logic remains untouched.
    """
    in_domain, info = domain_check(query)
    print("🧭 Domain check:", in_domain, "|", info)

    if not in_domain:
        return {
            "answer": (
                "❌ Out of scope for this knowledge base (insufficient domain coverage).\n\n"
                "Try:\n"
                "• Rephrase with more specific medical context\n"
                "• Ask about a condition/topic covered in the KB\n"
                "• Provide a source excerpt to add to the KB"
            ),
            "ood": True,
            "ood_info": info
        }

    # ✅ If domain check passes -> run your unchanged RAG
    resp = answer_question(query, top_k=TOP_K)
    return {**resp, "ood": False, "ood_info": info}

✅ Loaded scope index. Passages in scope index: 4127


In [15]:
# Cell 8 - Example usage

q = "What is hiv?"  # replace with your test question

resp = run_pipeline_with_domain_gate(q)
print(resp["answer"])
print("Answer:\n", resp["answer"])
print("\nRetrieved sources (top):")
for r in resp["retrieved"]:
    print("-", r["id"], "| score:", round(r["score"], 4))
    print("  excerpt:", (r["text"][:300] + "...") if len(r["text"])>300 else r["text"])

Token indices sequence length is longer than the specified maximum sequence length for this model (895 > 512). Running this sequence through the model will result in indexing errors


🧭 Domain check: True | {'reason': 'in_domain', 'top1': 0.5622984170913696, 'avg_topk': 0.46654750108718873, 'cohesion': 0.3967608720064163, 'scores_top5': [0.5622984170913696, 0.5320631265640259, 0.49664658308029175, 0.4733412265777588, 0.4445987939834595]}


/usr/local/lib/python3.12/dist-packages/transformers/generation/configuration_utils.py:567: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.0` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(


HIV-immune complex kidney disease (HIVICK) is still the most common cause of kidney disease in HIV-1-infected children and adolescents in many parts of the world, but there are exceptions where other types of kidney disease, such as HIV-immune complex kidney disease (HIVICK), are more prevalent.
Answer:
 HIV-immune complex kidney disease (HIVICK) is still the most common cause of kidney disease in HIV-1-infected children and adolescents in many parts of the world, but there are exceptions where other types of kidney disease, such as HIV-immune complex kidney disease (HIVICK), are more prevalent.

Retrieved sources (top):
- doc_2230 | score: 0.5514
  excerpt: Q: How does HIVICK (HIV immune complex kidney disease) develop? A: HIVICK is thought to develop either by the trapping or deposition of circulating immune complexes in the kidney parenchyma, or by in situ immune complex formation. These immune complexes consist of various HIV core and envelope antig...
- doc_417 | score: 0.4892
  e

In [16]:
!pip install -q numpy==1.26.4

In [17]:
!pip install -q spacy==3.6.1
!pip install -q scispacy==0.5.4

  error: subprocess-exited-with-error
  
  × pip subprocess to install build dependencies did not run successfully.
  │ exit code: 1
  ╰─> See above for output.
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
  Installing build dependencies ... error
error: subprocess-exited-with-error

× pip subprocess to install build dependencies did not run successfully.
│ exit code: 1
╰─> See above for output.

note: This error originates from a subprocess, and is likely not a problem with pip.
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  error: subprocess-exited-with-error
  
  × Preparing metadata (pyproject.toml) did not run successfully.
  │ exit code: 1
  ╰─> See above for output.
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
  Preparing metadata (pyproject.toml) ... error
error: metadata-generation-failed

× Encountered 

In [18]:



!pip install -q https://s3-us-west-2.amazonaws.com/ai2-s2-scispacy/releases/v0.5.4/en_core_sci_sm-0.5.4.tar.gz
!pip install -q https://s3-us-west-2.amazonaws.com/ai2-s2-scispacy/releases/v0.5.4/en_ner_bc5cdr_md-0.5.4.tar.gz


  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done


In [19]:
import spacy

nlp_sci = spacy.load("en_core_sci_sm")
nlp_bc5cdr = spacy.load("en_ner_bc5cdr_md")

print("✅ SciSpacy models loaded successfully")



/usr/local/lib/python3.12/dist-packages/spacy/language.py:2195: FutureWarning: Possible set union at position 6328
  deserializers["tokenizer"] = lambda p: self.tokenizer.from_disk(  # type: ignore[union-attr]


✅ SciSpacy models loaded successfully


In [20]:
nlp_base = spacy.load("en_core_web_sm")



/usr/local/lib/python3.12/dist-packages/spacy/util.py:910: UserWarning: [W095] Model 'en_core_web_sm' (3.8.0) was trained with spaCy v3.8.0 and may not be 100% compatible with the current version (3.7.5). If you see errors or degraded performance, download a newer compatible model or retrain your custom model with the current spaCy version. For more details and available updates, run: python -m spacy validate
  warnings.warn(warn_msg)


In [21]:
from transformers import pipeline

biomed_ner = pipeline(
    "ner",
    model="d4data/biomedical-ner-all",
    aggregation_strategy="simple"
)

print("✅ Transformer Biomedical NER loaded")


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/266M [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/373 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

/usr/local/lib/python3.12/dist-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(
Hardware accelerator e.g. GPU is available in the environment, but no `device` argument is passed to the `Pipeline` object. Model will be on CPU.


✅ Transformer Biomedical NER loaded


In [22]:
rag_output = resp["answer"]   # from your RAG pipeline


In [23]:
def split_sentences_with_ids(text):
    doc = nlp_sci(text)
    sents = [sent.text.strip() for sent in doc.sents]
    return [{"sent_id": i, "text": s} for i, s in enumerate(sents)]


claims_with_ids = split_sentences_with_ids(rag_output)



In [24]:
def extract_entities_multi_ner(text):
    entities = []

    # 1️⃣ SciSpacy BC5CDR (diseases, chemicals)
    doc = nlp_bc5cdr(text)
    for ent in doc.ents:
        entities.append({
            "text": ent.text,
            "label": ent.label_
        })

    # 2️⃣ Transformer Biomedical NER (more general coverage)
    for ent in biomed_ner(text):
        entities.append({
            "text": ent["word"],
            "label": ent["entity_group"]
        })

    return entities


In [25]:
def merge_entities(entities):
    seen = {}
    for e in entities:
        key = e["text"].lower()  # case-insensitive merge
        if key not in seen:
            seen[key] = e
    return list(seen.values())


In [26]:
RISK_MAP = {
    # SciSpacy BC5CDR
    "DISEASE": "HIGH",
    "CHEMICAL": "HIGH",

    # Common biomedical NER labels you may see
    "DRUG": "HIGH",
    "MEDICATION": "HIGH",
    "TREATMENT": "HIGH",
    "PROCEDURE": "MEDIUM",
    "SYMPTOM": "MEDIUM",
    "TEST": "MEDIUM",
    "ANATOMY": "LOW",

    # Often less “actionable clinical risk”
    "GENE": "LOW",
    "PROTEIN": "LOW",
    "CELL_LINE": "LOW",
    "UNKNOWN": "LOW"
}


In [27]:
import re

DOSE_REGEX = re.compile(r"\b(\d+(\.\d+)?)\s*(mg|mcg|µg|g|ml|mL|IU|units|%)\b", re.IGNORECASE)
FREQ_REGEX = re.compile(r"\b(once|twice|thrice)\s+(daily|a day)|\b(q\d+h|qd|bid|tid|qid)\b", re.IGNORECASE)
SAFETY_REGEX = re.compile(r"\b(contraindicated|adverse|toxicity|overdose|interaction|avoid|warning|monitor)\b", re.IGNORECASE)

In [28]:
final_results = []

for item in claims_with_ids:
    claim = item["text"]
    sent_id = item["sent_id"]

    raw_entities = extract_entities_multi_ner(claim)
    merged_entities = merge_entities(raw_entities)

    for e in merged_entities:
        e["risk"] = RISK_MAP.get(e["label"], "LOW")

    final_results.append({
        "sent_id": sent_id,        # ✅ keep sentence ID
        "claim": claim,
        "entities": merged_entities
    })


In [29]:
def score_sentence_risk(sentence: str, entities: list):
    score = 0
    reasons = []

    # A) entity label-based risk
    for e in entities:
        label = e.get("label", "UNKNOWN")
        risk = RISK_MAP.get(label, "LOW")

        if risk == "HIGH":
            score += 30
            reasons.append(f"HIGH label: {label} ({e.get('text')})")
        elif risk == "MEDIUM":
            score += 15
            reasons.append(f"MEDIUM label: {label} ({e.get('text')})")

    # B) heuristic safety triggers
    if DOSE_REGEX.search(sentence):
        score += 40
        reasons.append("Dose pattern (mg/mcg/ml/IU/%)")

    if FREQ_REGEX.search(sentence):
        score += 20
        reasons.append("Frequency pattern (qd/bid/tid/q12h/once daily)")

    if SAFETY_REGEX.search(sentence):
        score += 40
        reasons.append("Safety keyword (contraindicated/adverse/toxicity/interaction/...)")

    # cap score
    score = min(score, 100)

    return score, reasons

In [30]:
RISK_THRESHOLD = 20  # start with 60; lower if you want higher recall

hallucination_candidates = []
all_scored = []

for item in final_results:
    sent_id = item["sent_id"]
    sentence = item["claim"]
    entities = item["entities"]

    risk_score, reasons = score_sentence_risk(sentence, entities)

    scored_item = {
        "sent_id": sent_id,
        "sentence": sentence,
        "entities": entities,
        "risk_score": risk_score,
        "reasons": reasons
    }
    all_scored.append(scored_item)

    if risk_score >= RISK_THRESHOLD:
        hallucination_candidates.append(scored_item)

print(f"📝 Total sentences: {len(all_scored)}")
print(f"🚨 Routed HIGH-risk (score ≥ {RISK_THRESHOLD}): {len(hallucination_candidates)}\n")

for c in hallucination_candidates[:10]:
    print(f"[{c['sent_id']}] score={c['risk_score']} | {c['sentence']}")
    print(" reasons:", "; ".join(c["reasons"][:3]))
    print()

📝 Total sentences: 1
🚨 Routed HIGH-risk (score ≥ 20): 1

[0] score=60 | HIV-immune complex kidney disease (HIVICK) is still the most common cause of kidney disease in HIV-1-infected children and adolescents in many parts of the world, but there are exceptions where other types of kidney disease, such as HIV-immune complex kidney disease (HIVICK), are more prevalent.
 reasons: HIGH label: DISEASE (kidney disease); HIGH label: DISEASE (HIVICK)



In [31]:
print(len(hallucination_candidates))

1


In [32]:
# Reuse your existing SciSpacy sentence splitter
original_sentences = split_sentences_with_ids(rag_output)


print(f"📝 Total sentences in original RAG answer: {len(original_sentences)}\n")
for s in original_sentences:
    print(f"{s['sent_id']}. {s['text']}")


📝 Total sentences in original RAG answer: 1

0. HIV-immune complex kidney disease (HIVICK) is still the most common cause of kidney disease in HIV-1-infected children and adolescents in many parts of the world, but there are exceptions where other types of kidney disease, such as HIV-immune complex kidney disease (HIVICK), are more prevalent.


In [33]:

def semantic_similarity_check(claim_sentence: str, source_texts: List[str], top_k: int = 1) -> Dict[str, Any]:
    """
    Compare a high-risk claim sentence against a list of source documents.
    Returns the best matching source sentence and similarity score.
    """

    # Step 1: Split all source documents into sentences (using ID-based splitter)
    all_source_sents = []
    for doc in source_texts:
        sents = [s["text"] for s in split_sentences_with_ids(doc)]
        all_source_sents.extend(sents)

    if not all_source_sents:
        return {
            "claim": claim_sentence,
            "best_evidence": None,
            "similarity_score": 0.0

        }

    # Step 2: Compute embeddings
    claim_emb = embedder.encode([claim_sentence], convert_to_numpy=True, normalize_embeddings=True)
    source_embs = embedder.encode(all_source_sents, convert_to_numpy=True, normalize_embeddings=True)

    # Step 3: Compute cosine similarity
    sims = np.dot(source_embs, claim_emb.T).squeeze()

    # Step 4: Find best match
    best_idx = int(np.argmax(sims))
    best_score = float(sims[best_idx])
    best_sentence = all_source_sents[best_idx]

    return {
        "claim": claim_sentence,
        "best_evidence": best_sentence,
        "similarity_score": best_score

    }


In [34]:
from transformers import pipeline

NLI_MODEL = "cnut1648/biolinkbert-large-mnli-snli"

nli_pipeline = pipeline(
    "text-classification",
    model=NLI_MODEL,
    device=DEVICE
)

print("✅ NLI model loaded:", NLI_MODEL)



def normalize_nli_label(label: str) -> str:
    label = label.upper()
    if label in ("ENTAILMENT", "NEUTRAL", "CONTRADICTION"):
        return label

    mapping = {
        "LABEL_0": "ENTAILMENT",
        "LABEL_1": "NEUTRAL",
        "LABEL_2": "CONTRADICTION",
    }
    return mapping.get(label, label)


def nli_claim_verification(claim: str, evidence: str, max_len: int = 512):
    """
    Runs NLI with truncation to avoid >512 token crash in BERT models.
    Premise = evidence
    Hypothesis = claim
    """
    if evidence is None or str(evidence).strip() == "":
        return {"claim": claim, "evidence": evidence, "label": "NEUTRAL", "score": 0.0}

    input_text = f"Premise: {evidence}\nHypothesis: {claim}"

    result = nli_pipeline(
        input_text,
        truncation=True,
        max_length=max_len
    )[0]

    return {
        "claim": claim,
        "evidence": evidence,
        "label": normalize_nli_label(result["label"]),
        "score": float(result["score"])
    }


config.json:   0%|          | 0.00/979 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/1.33G [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/423 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

✅ NLI model loaded: cnut1648/biolinkbert-large-mnli-snli


In [35]:
# =========================
# NEW CELL: NLI label -> entailment probability
# (Must be defined BEFORE answer-level fallback cell)
# =========================

def nli_to_entail_prob(label: str, score: float) -> float:
    """Convert NLI output into a probability-like support score [0,1]."""
    lab = (label or "").upper()
    s = float(score or 0.0)

    if lab == "ENTAILMENT":
        return s
    if lab == "NEUTRAL":
        return 0.5 * s
    if lab == "CONTRADICTION":
        return 0.0
    return 0.0

In [36]:
# =========================
# Answer-level verification fallback (NLI -> Judge)
# Place this cell after nli_to_entail_prob() is defined
# =========================

ANSWER_NLI_SUPPORTED_TH = 0.80
ANSWER_NLI_UNSUPPORTED_TH = 0.45
MAX_EVIDENCE_CHARS = 1500  # adjust if needed

def build_evidence_pack(retrieved_texts: List[str], max_chars: int = MAX_EVIDENCE_CHARS) -> str:
    """
    Combine retrieved docs into one evidence pack, truncated to max_chars.
    """
    if not retrieved_texts:
        return ""
    joined = "\n\n".join([f"[Source {i+1}] {t}" for i, t in enumerate(retrieved_texts)])
    return joined[:max_chars]

def answer_level_nli_check(full_answer: str, retrieved_texts: List[str]) -> Dict[str, Any]:
    """
    Run ONE NLI check:
      Premise = combined retrieved evidence
      Hypothesis = full answer
    """
    evidence_pack = build_evidence_pack(retrieved_texts)

    # claim = full_answer, evidence = evidence_pack
    res = nli_claim_verification(full_answer, evidence_pack)
    entail_prob = nli_to_entail_prob(res["label"], res["score"])

    if entail_prob >= ANSWER_NLI_SUPPORTED_TH:
        status = "SUPPORTED"
    elif entail_prob < ANSWER_NLI_UNSUPPORTED_TH:
        status = "UNSUPPORTED"
    else:
        status = "UNCERTAIN"

    return {
        "evidence_pack": evidence_pack,
        "nli_label": res["label"],
        "nli_score": float(res["score"]),
        "entail_prob": float(entail_prob),
        "status": status
    }

def answer_level_verify(full_answer: str, retrieved_texts: List[str]) -> Dict[str, Any]:
    """
    Rule:
    - Run answer-level NLI
    - Only if NLI is UNCERTAIN, run LLM-as-Judge (1 call)
    """
    nli_res = answer_level_nli_check(full_answer, retrieved_texts)

    judge_res = None
    final_status = nli_res["status"]

    if nli_res["status"] == "UNCERTAIN":
        judge_res = llm_as_judge(full_answer, nli_res["evidence_pack"])
        final_status = "SUPPORTED" if judge_res["label"] == "SUPPORTED" else "UNSUPPORTED"

    return {
        "nli": nli_res,
        "judge": judge_res,
        "final_status": final_status
    }

In [37]:
# =========================
# Confidence + Decision Layer (FAST + FULL)
# =========================

# Fast thresholds (Similarity+NLI only)
FAST_VERIFIED_TH = 0.80
FAST_HALLUCINATED_TH = 0.45

# Full thresholds (Similarity+NLI+Judge)
FULL_VERIFIED_TH = 0.75
FULL_HALLUCINATED_TH = 0.50

# Regen should only happen when evidence is decent
SIM_FOR_REGEN = 0.60

def judge_to_score(judge_label: str) -> float:
    return 1.0 if (judge_label or "").upper() == "SUPPORTED" else 0.0

def confidence_fast(sim: float, nli_prob: float) -> float:
    return max(0.0, min(1.0, 0.55 * sim + 0.45 * nli_prob))

def confidence_full(sim: float, nli_prob: float, judge_score: float) -> float:
    return max(0.0, min(1.0, 0.40 * sim + 0.40 * nli_prob + 0.20 * judge_score))

def classify_fast(conf: float) -> str:
    if conf >= FAST_VERIFIED_TH:
        return "VERIFIED"
    if conf < FAST_HALLUCINATED_TH:
        return "HALLUCINATED"
    return "UNCERTAIN"

def classify_full(conf: float) -> str:
    if conf >= FULL_VERIFIED_TH:
        return "VERIFIED"
    if conf < FULL_HALLUCINATED_TH:
        return "HALLUCINATED"
    return "UNCERTAIN"

def has_very_high_risk_patterns(sentence: str) -> bool:
    if sentence is None:
        return False
    return bool(DOSE_REGEX.search(sentence) or SAFETY_REGEX.search(sentence))

In [38]:
def build_regeneration_prompt(claim: str, evidence: str):
    return (
        "You are a medical expert assistant.\n"
        "Rewrite the claim so that it is fully supported by the given evidence.\n"
        "Do NOT add new information.\n"
        "If the claim is incorrect, correct it using the evidence.\n\n"
        f"EVIDENCE:\n{evidence}\n\n"
        f"CLAIM:\n{claim}\n\n"
        "REVISED CLAIM:"
    )

In [39]:
def regenerate_claim(claim: str, evidence: str, max_length: int = 128):
    prompt = build_regeneration_prompt(claim, evidence)
    outputs = rag_generator(
        prompt,
        max_length=max_length,
        do_sample=False
    )
    return outputs[0]["generated_text"].strip()


In [40]:
# =========================
# Main verification loop (2-stage confidence gating)  ✅ FULLY UPDATED
# =========================

results = []
retrieved_texts = [r["text"] for r in resp["retrieved"]]

# ✅ NEW: Answer-level fallback result holder (used when no high-risk sentences)
answer_level_result = None

# ==========================================================
# CASE 0: No high-risk sentences -> Answer-level NLI first, Judge only if UNCERTAIN
# ==========================================================
if len(hallucination_candidates) == 0:
    answer_level_result = answer_level_verify(resp["answer"], retrieved_texts)

    print("✅ No high-risk sentences detected. Using Answer-level verification fallback.")
    print("   NLI status:", answer_level_result["nli"]["status"],
          "| entail_prob:", round(answer_level_result["nli"]["entail_prob"], 4),
          "| nli_label:", answer_level_result["nli"]["nli_label"],
          "| nli_score:", round(answer_level_result["nli"]["nli_score"], 4))
    if answer_level_result["judge"] is not None:
        print("   Judge used:", True, "| judge_label:", answer_level_result["judge"]["label"])
    else:
        print("   Judge used:", False)

# ==========================================================
# CASE 1: High-risk sentences exist -> run existing sentence-level verification loop
# ==========================================================
else:
    for item in hallucination_candidates:
        sent_id = item["sent_id"]
        claim_sent = item["sentence"]

        # --- Step A: Similarity (evidence selection) ---
        sim_result = semantic_similarity_check(claim_sent, retrieved_texts)
        evidence = sim_result["best_evidence"]
        sim_score = float(sim_result["similarity_score"])

        # --- Step B: NLI (fast) ---
        nli_res = nli_claim_verification(claim_sent, evidence)
        nli_prob = nli_to_entail_prob(nli_res["label"], nli_res["score"])

        # --- Step C: Fast confidence decision (Similarity+NLI) ---
        conf_fast = confidence_fast(sim_score, nli_prob)
        status_fast = classify_fast(conf_fast)

        judge_res = None
        conf_full = None
        status_full = None

        # Default: keep original
        final_claim = claim_sent
        final_status = status_fast
        final_conf = conf_fast
        regen_status = None

        # ==========================================================
        # CASE 1: FAST VERIFIED -> accept without judge
        # ==========================================================
        if status_fast == "VERIFIED":
            pass

        # ==========================================================
        # CASE 2: FAST HALLUCINATED -> regen/remove (no judge)
        # ==========================================================
        elif status_fast == "HALLUCINATED":
            if evidence is not None and sim_score >= SIM_FOR_REGEN:
                regenerated = regenerate_claim(claim_sent, evidence)

                # Re-check regenerated (fast first)
                nli2 = nli_claim_verification(regenerated, evidence)
                nli2_prob = nli_to_entail_prob(nli2["label"], nli2["score"])
                conf2_fast = confidence_fast(sim_score, nli2_prob)
                status2_fast = classify_fast(conf2_fast)

                # If still uncertain, only then judge (optional)
                if status2_fast == "UNCERTAIN" and has_very_high_risk_patterns(regenerated):
                    judge2 = llm_as_judge(regenerated, evidence)
                    conf2_full = confidence_full(sim_score, nli2_prob, judge_to_score(judge2["label"]))
                    status2_full = classify_full(conf2_full)
                else:
                    judge2 = None
                    conf2_full = None
                    status2_full = None

                # Accept regen only if VERIFIED (strict medical safety)
                if status2_fast == "VERIFIED" or status2_full == "VERIFIED":
                    final_claim = regenerated
                    final_status = "REGEN_VERIFIED"
                    final_conf = conf2_full if (status2_full == "VERIFIED") else conf2_fast
                    regen_status = "REGEN_ACCEPTED"
                else:
                    # Keep original but mark that it remained hallucinated/unsafe
                    final_status = "HALLUCINATED_UNFIXED"
                    regen_status = "REGEN_REJECTED"
            else:
                # Not enough evidence to safely regenerate
                final_status = "HALLUCINATED_NO_EVIDENCE"
                regen_status = "REGEN_SKIPPED"

        # ==========================================================
        # CASE 3: FAST UNCERTAIN -> judge only if very high-risk patterns
        # ==========================================================
        else:  # UNCERTAIN
            if has_very_high_risk_patterns(claim_sent):
                judge_res = llm_as_judge(claim_sent, evidence)
                conf_full = confidence_full(sim_score, nli_prob, judge_to_score(judge_res["label"]))
                status_full = classify_full(conf_full)

                final_conf = conf_full
                final_status = status_full

                # If still hallucinated after judge -> regen/remove
                if status_full == "HALLUCINATED":
                    if evidence is not None and sim_score >= SIM_FOR_REGEN:
                        regenerated = regenerate_claim(claim_sent, evidence)

                        # Re-check regenerated (fast)
                        nli2 = nli_claim_verification(regenerated, evidence)
                        nli2_prob = nli_to_entail_prob(nli2["label"], nli2["score"])
                        conf2_fast = confidence_fast(sim_score, nli2_prob)
                        status2_fast = classify_fast(conf2_fast)

                        # Judge regenerated only if still uncertain + very high-risk patterns
                        if status2_fast == "UNCERTAIN" and has_very_high_risk_patterns(regenerated):
                            judge2 = llm_as_judge(regenerated, evidence)
                            conf2_full = confidence_full(sim_score, nli2_prob, judge_to_score(judge2["label"]))
                            status2_full = classify_full(conf2_full)
                        else:
                            judge2 = None
                            conf2_full = None
                            status2_full = None

                        if status2_fast == "VERIFIED" or status2_full == "VERIFIED":
                            final_claim = regenerated
                            final_status = "REGEN_VERIFIED"
                            final_conf = conf2_full if (status2_full == "VERIFIED") else conf2_fast
                            regen_status = "REGEN_ACCEPTED"
                        else:
                            final_status = "HALLUCINATED_UNFIXED"
                            regen_status = "REGEN_REJECTED"
                    else:
                        final_status = "HALLUCINATED_NO_EVIDENCE"
                        regen_status = "REGEN_SKIPPED"
            else:
                # UNCERTAIN but not very high-risk -> keep but flag (no judge)
                final_status = "UNCERTAIN_LOW_PRIORITY"

        results.append({
            "sent_id": sent_id,
            "original_claim": claim_sent,
            "final_claim": final_claim,
            "evidence": evidence,
            "similarity_score": sim_score,
            "nli_label": nli_res["label"],
            "nli_score": float(nli_res["score"]),
            "fast_confidence": conf_fast,
            "fast_status": status_fast,
            "judge_label": (judge_res["label"] if judge_res else None),
            "final_confidence": final_conf,
            "final_status": final_status,
            "regen_status": regen_status
        })

    print("✅ results length:", len(results))
    for r in results[:10]:
        print(r)

✅ results length: 1
{'sent_id': 0, 'original_claim': 'HIV-immune complex kidney disease (HIVICK) is still the most common cause of kidney disease in HIV-1-infected children and adolescents in many parts of the world, but there are exceptions where other types of kidney disease, such as HIV-immune complex kidney disease (HIVICK), are more prevalent.', 'final_claim': 'HIV-immune complex kidney disease (HIVICK) is still the most common cause of kidney disease in HIV-1-infected children and adolescents in many parts of the world, but there are exceptions where other types of kidney disease, such as HIV-immune complex kidney disease (HIVICK), are more prevalent.', 'evidence': 'HIV-associated nephropathy (HIVAN) is still the most common cause of kidney disease in HIV-1-infected children and adolescents in many parts of the world, but there are exceptions where other types of kidney disease, such as HIV-immune complex kidney disease (HIVICK), are more prevalent.', 'similarity_score': 0.905661

In [41]:
def rebuild_final_answer_by_id(original_answer: str, results):
    """
    Rebuild final answer using sentence IDs.
    Replace sentences only when final_claim differs from original_claim.
    """
    original_sents = split_sentences_with_ids(original_answer)

    # Map sent_id -> final_claim only when changed
    replace_map = {
        r["sent_id"]: r["final_claim"]
        for r in results
        if r.get("final_claim") is not None and r.get("final_claim") != r.get("original_claim")
    }

    final_sents = []
    for s in original_sents:
        sent_id = s["sent_id"]
        final_sents.append(replace_map.get(sent_id, s["text"]))

    return " ".join(final_sents)


# ✅ Generate final verified answer (ID-based reconstruction)
DISCLAIMER = (
    "⚠️ Disclaimer: Some high-risk medical statements were automatically verified and, "
    "where necessary, revised for safety. This output is for informational purposes only "
    "and is not medical advice. Please consult a qualified healthcare professional before "
    "making clinical decisions.\n\n"
)

# ✅ NEW: Disclaimer for answer-level fallback failure
ANSWER_LEVEL_DISCLAIMER = (
    "⚠️ Disclaimer: No high-risk entities were detected, but a full-response verification "
    "step found that the overall answer may not be fully supported by the retrieved evidence. "
    "Please verify with reliable medical sources or consult a qualified healthcare professional.\n\n"
)

# 1) Rebuild final answer from sentence-level results (if any)
final_answer = rebuild_final_answer_by_id(
    original_answer=resp["answer"],
    results=results
)

# 2) Sentence-level intervention detection (existing logic)
intervention_detected = any(
    r["final_status"] in [
        "REGEN_VERIFIED",
        "HALLUCINATED_UNFIXED",
        "UNCERTAIN",
        "UNCERTAIN_LOW_PRIORITY"
    ]
    for r in results
)

# 3) ✅ NEW: answer-level intervention detection (only when no high-risk sentences)
answer_level_intervention = False
if 'answer_level_result' in globals() and answer_level_result is not None:
    # If the single-step whole-answer verification says UNSUPPORTED, warn user
    if answer_level_result.get("final_status") == "UNSUPPORTED":
        answer_level_intervention = True

# 4) Apply disclaimers (sentence-level and/or answer-level)
if intervention_detected:
    final_answer = DISCLAIMER + final_answer

if answer_level_intervention:
    final_answer = ANSWER_LEVEL_DISCLAIMER + final_answer

print("✅ FINAL VERIFIED ANSWER:\n")
print(final_answer)

# ✅ OPTIONAL: print fallback verification summary (useful for debugging/reporting)
if 'answer_level_result' in globals() and answer_level_result is not None:
    print("\n📌 Answer-level verification summary:")
    print(" - final_status:", answer_level_result.get("final_status"))
    print(" - nli_status:", answer_level_result["nli"]["status"])
    print(" - entail_prob:", round(answer_level_result["nli"]["entail_prob"], 4))
    if answer_level_result["judge"] is not None:
        print(" - judge_used: True | judge_label:", answer_level_result["judge"]["label"])
    else:
        print(" - judge_used: False")

✅ FINAL VERIFIED ANSWER:

HIV-immune complex kidney disease (HIVICK) is still the most common cause of kidney disease in HIV-1-infected children and adolescents in many parts of the world, but there are exceptions where other types of kidney disease, such as HIV-immune complex kidney disease (HIVICK), are more prevalent.
